# Proyecto 2 — Extracción de relaciones clínicas por aprendizaje

**Demostración ejecutable.** Este cuaderno recorre, paso a paso y con datos reales, cómo el
Proyecto 2 sustituye las reglas escritas a mano del Proyecto 1 por un modelo aprendido.

La idea en una frase: **mantenemos todo igual y cambiamos solo quién decide.**

```
texto clínico
     ↓
entidades (gazetteer UMLS)                    ← igual que el P1
     ↓
el P1 propone 2.377 pares candidatos          ← igual que el P1
     ↓
  ┌──────────────┴──────────────┐
P1: reglas a mano          P2: pesos aprendidos de 413 ejemplos
  └──────────────┬──────────────┘
     ↓
grafo de conocimiento
```

Todo lo que hace falta está en el repositorio: no se necesita el gazetteer UMLS (289 MB)
porque las entidades ya están extraídas en `data/processed/entities.csv`.

## 0. Preparación

En local, ejecuta el cuaderno desde la raíz del repositorio. En Colab, descomenta las dos
líneas del clon.

In [ ]:
# --- Colab: descomenta estas dos líneas ---
# !git clone -q --branch prj2/relation-classifier https://github.com/JaocHatter/Projeto-de-Natural-Language-Processing---Knowledge-Graph.git
# %cd Projeto-de-Natural-Language-Processing---Knowledge-Graph

import sys, os
from pathlib import Path

root = Path.cwd()
while not (root / "src" / "clinical_kg").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root / "src"))
os.chdir(root)
print("raíz del proyecto:", root)

## 1. El corpus

50 relatos clínicos de PubMed Central, ya limpiados por el Proyecto 1.

In [ ]:
import csv, collections

read = lambda p: list(csv.DictReader(open(p, newline="", encoding="utf-8", errors="replace")))
cases    = read("data/interim/cases_clean.csv")
entities = read("data/processed/entities.csv")
gold     = read("data/interim/gold_relations.csv")

print(f"relatos clínicos : {len(cases)}")
print(f"entidades        : {len(entities):,} en {len({e['cui'] for e in entities})} conceptos UMLS distintos")
print(f"candidatos juzgados a mano : {len(gold)} "
      f"({sum(1 for g in gold if g['is_relation']=='1')} son relación, "
      f"{sum(1 for g in gold if g['is_relation']=='0')} no)")

### Una frase concreta

La usaremos durante todo el cuaderno.

In [ ]:
case = next(c for c in cases if c["case_id"] == "PMC3437073_01")
text = case["case_text"]
i = text.find("Ultrasound of the abdomen")
frase = text[i:i+78]
print(frase)
print()
for e in entities:
    if e["case_id"] == case["case_id"] and i <= int(e["start"]) < i + 78:
        print(f"  {e['surface_text']:<18} {e['entity_type']:<10} {e['cui']}")

Las etiquetas (*Exam*, *BodyPart*, *Finding*…) no las inventamos: salen del **tipo semántico
que UMLS asigna a cada concepto**. Esa información alimentará después al clasificador.

## 2. Los candidatos — esto NO cambia

El generador del Proyecto 1 propone todo par de entidades a menos de 20 palabras. Es la
pieza que mantenemos fija: así los dos sistemas ven exactamente los mismos pares y la
comparación es limpia.

In [ ]:
from clinical_kg.relations.extract import analyze_case, load_entities
from clinical_kg.paths import DEFAULT_ENTITIES

ents_by_case = load_entities(DEFAULT_ENTITIES)
candidatos = analyze_case(text, ents_by_case[case["case_id"]], {}, threshold=float("-inf"))

print(f"candidatos en este relato: {len(candidatos)}\n")
for rel in candidatos:
    if rel["sent_start"] <= i < rel["sent_end"]:
        izq = rel["head"]["surface_text"] if rel["head"] else "PACIENTE"
        print(f"  {izq:<22} → {rel['tail']['surface_text']:<20} "
              f"disparador={' '.join(rel['trigger']['words']) if rel['trigger'] else '—'}")

## 3. Describir cada par: 62 pistas

Aquí empieza el Proyecto 2. Cada candidato se convierte en una lista de pistas.

In [ ]:
from clinical_kg.learning.dataset import load_examples

ejemplos = load_examples(("struct", "assoc"))
print(f"{len(ejemplos)} candidatos juzgados, con "
      f"{len({n for e in ejemplos for n in e.features})} pistas distintas\n")

ej = next(e for e in ejemplos if e.head_text == "abdomen" and "spleen" in e.tail_text)
print(f"EJEMPLO: {ej.head_text} → {ej.tail_text}   (¿es relación? {'sí' if ej.is_relation else 'no'})\n")
for nombre, valor in sorted(ej.features.items()):
    print(f"  {nombre:<28} {valor}")

Las pistas `pair:…`, `head_type:…` y `tail_type:…` **vienen de UMLS**: son 51 de las 62.
Las tres `pmi_*` y `llr_max` miden si las palabras intermedias suelen ir juntas en el corpus.

## 4. Los n-gramas — el detalle que importa

El enunciado exige n-gramas. Lo intuitivo es meterlos como pistas sueltas (`of the`,
`treated with`, una por cada uno). **Lo probamos y empeora**: con 413 ejemplos no hay
material para aprender ~2.900 parámetros.

La alternativa es **comprimirlos**: entrenamos dos modelos de lenguaje, uno con los textos
que sí expresan relación y otro con los que no, y preguntamos cuál considera más probable
el texto que separa las dos entidades.

In [ ]:
from clinical_kg.learning.ngram import NgramLM, tokenize

si_relacion = [e.context for e in ejemplos if e.is_relation]
no_relacion = [e.context for e in ejemplos if not e.is_relation]

modelo_si = NgramLM().fit(si_relacion)
modelo_no = NgramLM().fit(no_relacion)

print(f"entrenados con {len(si_relacion)} contextos positivos y {len(no_relacion)} negativos\n")
print(f"{'texto entre las entidades':<30}{'modelo SÍ':>12}{'modelo NO':>12}   veredicto")
for t in ["revealed", "was treated with", "of the", "and", "caused by", "indicated an"]:
    doc = tokenize(t)
    a, b = modelo_si.log_probability(doc), modelo_no.log_probability(doc)
    print(f"  {t!r:<28}{a:>12.3f}{b:>12.3f}   {'RELACIÓN' if a > b else 'no'}")

Fíjate en que el modelo aprendió solo que `and` indica relación: son las enumeraciones
(*"fiebre, tos **and** malestar"*), que el Proyecto 1 trataba con una regla aparte escrita
a mano.

El suavizado es imprescindible aquí: **el 81,6 % de los bigramas del corpus aparece una sola
vez**, así que sin suavizar casi todo tendría probabilidad cero.

In [ ]:
import math
doc = tokenize("xyzzy frobnicate")       # dos palabras que no existen en el corpus
print(f"log-prob de un texto nunca visto: {modelo_si.log_probability(doc):.3f}")
print("(finito gracias al suavizado add-k; sin él sería -infinito y el modelo se rompería)")

## 5. Entrenar y comparar con el Proyecto 1

**Validación dejando un caso fuera**: 10 rondas; en cada una el modelo aprende con 9 relatos
y se examina con el décimo, que nunca ha visto. Así ningún número está medido sobre texto
que el modelo ya leyó.

In [ ]:
from clinical_kg.learning.experiment import folds, with_lm, run_p1_baseline, evaluate_binary, prf
from clinical_kg.learning.models import OneVsRest

# --- línea base: el Proyecto 1 tal cual ---
pred_p1, _ = run_p1_baseline(ejemplos, "binary")
p1 = prf(*evaluate_binary(pred_p1))

# --- el Proyecto 2 ---
pred_p2, por_caso = [], []
for caso, train, test in folds(ejemplos):
    X_tr, X_te = with_lm(train, test, 1.0, None)          # la pista n-grama, sin fuga
    modelo = OneVsRest().fit(X_tr, [e.relation or "NONE" for e in train])
    ronda = [(e.is_relation, 0 if modelo.predict(x) == "NONE" else 1)
             for e, x in zip(test, X_te)]
    pred_p2 += ronda
    por_caso.append((caso, prf(*evaluate_binary(ronda))[2]))
p2 = prf(*evaluate_binary(pred_p2))

print(f"{'':<26}{'precisión':>11}{'cobertura':>11}{'F1':>8}")
print(f"  {'Proyecto 1 (reglas)':<24}{p1[0]:>11.3f}{p1[1]:>11.3f}{p1[2]:>8.3f}")
print(f"  {'Proyecto 2 (aprendido)':<24}{p2[0]:>11.3f}{p2[1]:>11.3f}{p2[2]:>8.3f}")
print(f"\n  mejora: {p2[2] - p1[2]:+.3f}")

In [ ]:
print("F1 relato a relato (para ver la variabilidad, no solo la media):\n")
for caso, f1 in sorted(por_caso, key=lambda x: -x[1]):
    print(f"  {caso:<20} {f1:.3f}  {'█' * int(f1 * 40)}")
media = sum(f for _, f in por_caso) / len(por_caso)
desv = (sum((f - media)**2 for _, f in por_caso) / len(por_caso)) ** 0.5
print(f"\n  media {media:.3f} ± {desv:.3f}")

## 6. El grafo: mismo trabajo, grafos distintos

El enunciado pide analizar el grafo resultante, no solo el F1. Dos métodos con rendimiento
parecido pueden producir grafos muy diferentes.

In [ ]:
from clinical_kg.graph.model import build_graph, CLINICAL_RELATIONS

medidas = read("data/processed/measurements.csv")
def estadisticas(path):
    rels = collections.defaultdict(list)
    for r in read(path): rels[r["case_id"]].append(r)
    por_tipo, aristas = collections.Counter(), set()
    for c in cases:
        g = build_graph(c, c["case_text"],
                        [e for e in entities if e["case_id"] == c["case_id"]],
                        [m for m in medidas if m["case_id"] == c["case_id"]],
                        relations=rels.get(c["case_id"], []))
        for e in g["edges"]:
            if e["relation"] in CLINICAL_RELATIONS:
                por_tipo[e["relation"]] += 1
                aristas.add((c["case_id"], e["source"], e["target"], e["relation"]))
    return por_tipo, aristas

p1_tipos, p1_aristas = estadisticas("data/processed/relations.csv")
p2_tipos, p2_aristas = estadisticas("data/processed/relations_ml.csv")

print(f"{'relación':<20}{'P1':>8}{'P2':>8}")
for r in sorted(set(p1_tipos) | set(p2_tipos)):
    print(f"  {r:<18}{p1_tipos[r]:>8}{p2_tipos[r]:>8}")
print(f"\n  total          {sum(p1_tipos.values()):>8}{sum(p2_tipos.values()):>8}")
print(f"  en común       {len(p1_aristas & p2_aristas):>8}")

Solo coinciden unas 591 aristas de las ~1.400 de cada uno. **Mismo corpus, mismos candidatos,
rendimiento parecido — y sin embargo grafos sustancialmente distintos.** Ese contraste es
uno de los resultados del proyecto.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

tipos = sorted(set(p1_tipos) | set(p2_tipos))
x = np.arange(len(tipos)); w = 0.38
fig, ax = plt.subplots(figsize=(10, 4.2))
ax.bar(x - w/2, [p1_tipos[t] for t in tipos], w, label="P1 — reglas a mano", color="#8b96a0")
ax.bar(x + w/2, [p2_tipos[t] for t in tipos], w, label="P2 — aprendido", color="#0f6d6a")
ax.set_xticks(x); ax.set_xticklabels(tipos, rotation=35, ha="right", fontsize=9)
ax.set_ylabel("aristas en el grafo"); ax.legend(frameon=False)
ax.set_title("Qué relaciones encuentra cada método", loc="left", fontsize=11)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

## 7. Qué probamos que **no** funcionó

Vale tanto como lo que sí: enseña dónde el método no sirve.

| Idea | Efecto en F1 | Por qué |
|---|---|---|
| N-gramas como ~2.900 pistas sueltas | 0,738 → 0,677 | 413 ejemplos no sostienen 3.000 parámetros |
| Definiciones de UMLS entrenando los modelos de lenguaje | 0,730 → 0,715 | unen conceptos con conectores genéricos (`and`, `of the`) |
| Definiciones de UMLS como fondo de los dos modelos | sin efecto | un fondo compartido se cancela en el cociente |
| Posición en el relato | 0,730 → 0,719 | separa *tipos* de relación, no relación de no-relación |
| Términos MeSH de los metadatos | inviable | solo 7 de 50 artículos los tienen |
| **Definiciones de UMLS detrás del PMI** | **0,730 → 0,738** | el PMI necesita más texto, no otro registro |

La última línea es la que sí funcionó: el mismo recurso, usado en otro sitio.

## 8. Conclusiones

1. **Aprender gana a escribir reglas**: 0,628 → 0,757 en F1, y acertar el tipo de relación
   pasa de 0,774 a 0,870.
2. **Los n-gramas hay que comprimirlos, no esparcirlos.** Como pistas sueltas empeoran;
   condensados en un modelo de lenguaje, ayudan.
3. **El modelo da igual, las pistas no.** Regresión logística, SVM y random forest empatan
   (0,723 / 0,725 / 0,727); lo que mueve el resultado es cómo describes el problema.
4. **El cuello de botella son los datos.** 50 relatos, y el 81,6 % de los bigramas aparece
   una sola vez. Ampliar el corpus es lo que más rendiría.

### Cómo reproducirlo entero

```bash
python -m clinical_kg learn-relations --task relation   # la comparación
python -m clinical_kg extract-relations-ml              # el extractor
python -m unittest discover -s tests                    # 96 tests
```